In [1]:
import json 
import os 
import time 
from typing import Literal

import ollama
import pandas as pd 
from dotenv import load_dotenv
from openai import ContentFilterFinishReasonError, OpenAI, RateLimitError
from pydantic import BaseModel, Field 
load_dotenv()
openrouter_api_key=os.getenv("OPENROUTER_API_KEY")
openrouter_base_url="https://openrouter.ai/api/v1"
openrouter=OpenAI(base_url=openrouter_base_url, api_key=openrouter_api_key)


In [2]:
from loader import load_json, load_all
claims_path="E:\\undercover\\data\\evals\\held_out_claims.json"
labels_path="E:\\undercover\\data\\evals\\held_out_decisions.json"
policy_path="E:\\undercover\\data\\policies.json"
claims=load_json(claims_path)
labels=load_json(labels_path)
policies=load_json(policy_path)

len(claims), len(labels), len(policies)

(25, 25, 50)

In [3]:
#set up the harness
INSTRUCTIONS=""" You ae a claims handler at Undercover Travel Insurance
working through a queue of new claims on your own.

Everything you need is in this folder:
- claims/has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named bu their ids
- policy_wording.md is the policy woprding and claims manual.md is the claims handling manual
- company.json has the companys limits

Decide every claim in claims/. Keep a to-do list with ohne task for each claim and give each claim to the handler-handler agent, which 
decides. When every claim has a decision, write handover.md: a short note for the handler on what was devided and anything they should know
"""

HANDLER_INSTUCTIONS=""" You are a claims handler at undercover travel insurance.
Decide the claim you are given, 
Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md
Choose one decision:
approve: the wording covers the claim and the required documents are attached. Workout the payout using the payment section
decline: the wording does not cover the claim.
refer: a colleage needs to look at it because a required document is missing or the information disagree with the documents provided

Write the decision to decisions/<claim_id>.json with the fields claim_id, deicion, payout(in dollars) and rationale
Then reply with one line: the claim id, the decision and the payout.
"""